# Camada Silver

Este notebook transforma os dados brutos da Bronze em dados tratados e padronizados.

Nesta camada são aplicadas as regras de qualidade e de negócio definidas na atividade, incluindo:

- padronização e tradução de valores;
- conversão segura de tipos;
- tratamento de valores inválidos;
- deduplicação;
- normalização de datas;
- criação de atributos derivados;
- tratamento de relações multivaloradas;
- preenchimento da série temporal da cotação.

As tabelas Bronze não são alteradas.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "olist_medallion"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")





DataFrame[]

## Funções auxiliares de qualidade

As funções abaixo centralizam as verificações de qualidade utilizadas nas diferentes tabelas Silver. Cada validação registra a tabela, a regra verificada, a quantidade de falhas e o resultado da checagem.

In [0]:
from pyspark.sql import Row
from datetime import datetime

# Armazena os resultados das validações executadas ao longo da camada Silver
resultados_qualidade = []

# Valida regras de qualidade linha a linha e registra a quantidade de falhas.
def dq_check(tabela, regra, df, condicao):
    total = df.count()

    falhas = df.filter(condicao.isNull() | ~condicao).count()
    aprovado = falhas == 0
    resultados_qualidade.append(Row(
        tabela=tabela, regra=regra, total_linhas=total,
        falhas=falhas, unidade_falha="linhas",
        aprovado=aprovado, verificado_em=datetime.now()
    ))
    print(f"{tabela} | {regra} | aprovado={aprovado} | falhas={falhas}/{total}")

# Valida a unicidade de uma ou mais colunas utilizadas como chave.
def dq_check_unique(tabela, regra, df, colunas_chave):
    total = df.count()
    chaves_duplicadas = (
        df.groupBy(*colunas_chave).count().filter("count > 1").count()
    )
    aprovado = chaves_duplicadas == 0
    resultados_qualidade.append(Row(
        tabela=tabela, regra=regra, total_linhas=total,
        falhas=chaves_duplicadas, unidade_falha="chaves_duplicadas",
        aprovado=aprovado, verificado_em=datetime.now()
    ))
    print(f"{tabela} | {regra} | aprovado={aprovado} | chaves duplicadas={chaves_duplicadas}")

## 1. Informações dos filmes — `silver.tb_info_filmes`

A tabela é construída a partir de `bronze.tb_movies_info`.

São aplicadas as seguintes regras:

- manutenção apenas da ingestão mais recente de cada filme;
- normalização e tradução do status;
- conversão da duração para inteiro;
- tratamento de diferentes formatos de data;
- validação de datas de calendário;
- criação da coluna `ano_lancamento`.

In [0]:
df_info = spark.table(f"{catalog}.bronze.tb_movies_info")

# Deduplicação por filme: como a Bronze utiliza append, um mesmo filme
# pode existir em mais de uma ingestão. Mantém-se o registro mais recente.
janela_filme = Window.partitionBy("id").orderBy(
    F.col("ingestion_datetime").desc(), F.col("tconst"), F.col("title"), F.col("original_title"), F.col("original_language"), F.col("release_date"), F.col("runtime"), F.col("status"), F.col("overview"), F.col("tagline")
)
df_info = df_info.filter(F.col("id").isNotNull()).withColumn(
    "_ordem", F.row_number().over(janela_filme)
).filter(F.col("_ordem") == 1)

df_info = df_info.withColumn(

    # Normaliza caixa, espaços, hífens e underscores antes da tradução,
    # evitando que diferenças de formatação gerem categorias distintas.

    "status_limpo",
    F.lower(F.trim(F.regexp_replace("status", "[-_]+", " ")))
).withColumn(
    "status_limpo", F.regexp_replace("status_limpo", " +", " ")
).withColumn(

    # Traduz apenas os status pertencentes ao domínio definido.
    # Valores desconhecidos ou corrompidos são classificados como "Não Informado".

    "status_filme",
    F.when(F.col("status_limpo") == "released", "Lançado")
     .when(F.col("status_limpo") == "post production", "Pós-Produção")
     .when(F.col("status_limpo") == "in production", "Em Produção")
     .when(F.col("status_limpo") == "planned", "Planejado")
     .when(F.col("status_limpo") == "rumored", "Rumores")
     .when(F.col("status_limpo") == "canceled", "Cancelado")
     .otherwise("Não Informado")
)
df_info = df_info.withColumn(

    # A conversão é feita somente para valores estritamente numéricos;
    # conteúdos incompatíveis são tratados como ausentes.

    "duracao_minutos",
    F.when(F.trim("runtime").rlike("^[0-9]{1,8}$"),
           F.col("runtime").cast("int"))
     .otherwise(None)
)
df_info_preparada = df_info.select(
    F.col("id").cast("string").alias("id_filme"),
    F.trim("title").alias("titulo"),
    F.trim("original_title").alias("titulo_original"),
    "release_date",
    "duracao_minutos",
    F.lower(F.trim("original_language")).alias("idioma_original"),
    "status_filme",
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias("frase_divulgacao")
)
display(df_info_preparada.limit(10))

df_datas = df_info_preparada.withColumn("_data_texto", F.trim("release_date"))
df_datas = df_datas.withColumn(

    # Padroniza os diferentes formatos encontrados na origem para yyyy-MM-dd.
    # Valores que não correspondem a nenhum dos formatos esperados permanecem nulos.

    "_data_iso",
    F.when(
        F.col("_data_texto").rlike(r"^[0-9]{4}-[0-9]{2}-[0-9]{2}$"),
        F.col("_data_texto")
    ).when(
        F.col("_data_texto").rlike(r"^[0-9]{2}/[0-9]{2}/[0-9]{4}$"),
        F.regexp_replace("_data_texto", r"^([0-9]{2})/([0-9]{2})/([0-9]{4})$", "$3-$2-$1")
    ).when(
        F.col("_data_texto").rlike(r"^[0-9]{2}-[0-9]{2}-[0-9]{4}$"),
        F.regexp_replace("_data_texto", r"^([0-9]{2})-([0-9]{2})-([0-9]{4})$", "$3-$1-$2")
    ).otherwise(None)
)

df_datas = df_datas.withColumn(
    "_ano", F.regexp_replace("_data_iso", r"^([0-9]{4})-([0-9]{2})-([0-9]{2})$", "$1").cast("int")
).withColumn(
    "_mes", F.regexp_replace("_data_iso", r"^([0-9]{4})-([0-9]{2})-([0-9]{2})$", "$2").cast("int")
).withColumn(
    "_dia", F.regexp_replace("_data_iso", r"^([0-9]{4})-([0-9]{2})-([0-9]{2})$", "$3").cast("int")
)


df_datas = df_datas.withColumn(

# A validação manual de ano, mês e dia impede que datas de calendário
# impossíveis sejam aceitas silenciosamente durante a conversão.

    "_bissexto",
    (F.col("_ano") % 4 == 0) &
    ((F.col("_ano") % 100 != 0) | (F.col("_ano") % 400 == 0))
).withColumn(
    "_ultimo_dia",
    F.when(F.col("_mes").isin([4, 6, 9, 11]), 30)
     .when((F.col("_mes") == 2) & F.col("_bissexto"), 29)
     .when(F.col("_mes") == 2, 28)
     .otherwise(31)
).withColumn(
    "_data_valida",
    F.coalesce(
        (F.col("_ano") >= 1) &
        (F.col("_mes") >= 1) & (F.col("_mes") <= 12) &
        (F.col("_dia") >= 1) & (F.col("_dia") <= F.col("_ultimo_dia")),
        F.lit(False)
    )
)


df_datas = df_datas.withColumn(

    # Apenas datas válidas são convertidas para DATE.
    # O ano de lançamento é derivado da data já tratada.
    "data_lancamento",
    F.to_date(
        F.when(F.col("_data_valida"), F.col("_data_iso")).otherwise(None),
        "yyyy-MM-dd"
    )
).withColumn("ano_lancamento", F.year("data_lancamento"))

display(df_datas.filter(
    F.col("release_date").isNotNull() & F.col("data_lancamento").isNull()
).select("id_filme", "release_date").limit(20))

df_info_silver = df_datas.select(
    "id_filme", "titulo", "titulo_original", "data_lancamento",
    "duracao_minutos", "idioma_original", "status_filme",
    "sinopse", "frase_divulgacao", "ano_lancamento"
)
dq_check_unique("silver.tb_info_filmes", "ID único", df_info_silver, ["id_filme"])
dq_check("silver.tb_info_filmes", "ID não nulo", df_info_silver,
         F.col("id_filme").isNotNull())
dq_check("silver.tb_info_filmes", "status no domínio", df_info_silver,
         F.col("status_filme").isin([
             "Lançado", "Pós-Produção", "Em Produção", "Planejado",
             "Rumores", "Cancelado", "Não Informado"
         ]))
df_info_silver.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(f"{catalog}.silver.tb_info_filmes")


id_filme,titulo,titulo_original,release_date,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao
66213,Iruttu Araiyil Murattu Kuthu,இருட்டு அறையில் முரட்டு குத்து,2018-05-04,118,ta,Lançado,A sexually-starved spirit traps two couples in a bungalow in Bangkok where they have planned to stay for a week. The spirit wants one of the two men to have sex with her. Can they manage to escape?,null
68730,Silence,Silence,2016-12-22,161,en,Lançado,"Two Jesuit priests travel to seventeenth century Japan which has, under the Tokugawa shogunate, banned Catholicism and almost all foreign contact.",null
80765,Dimmu Borgir - Forces of the Northern Night,Dimmu Borgir - Forces of the Northern Night,2017-04-28,59,en,Lançado,01. Xibir 02. Born Treacherous 03. Gateways 04. Dimmu Borgir (orchestral) 05. Dimmu Borgir 06. Ritualist 07. A Jewel Traced Through Coal 08. Eradication Instincts Defined (orchestral) 09. Vredesbyrd 10. Progenies Of The Great Apocalypse 11. The Serpentine Offering 12. Fear And Wonder (orchestral) 13. Puritania 14. Kings Of The Carnival Creation 15. Mourning Palace 16. Perfection Or Vanity,null
115436,Hason Raja,Hason Raja,2017-03-31,165,bn,Lançado,"The story of Hason Raja, a flamboyant, ruthless zamindar from Sylhet who fell in love with Dilaram, who transformed him and later he became a poet, singing songs and wandering all across Bangladesh.",null
137116,Smurfs: The Lost Village,Smurfs: The Lost Village,2017-03-23,89,en,Lançado,"In this fully animated, all-new take on the Smurfs, a mysterious map sets Smurfette and her friends Brainy, Clumsy and Hefty on an exciting race through the Forbidden Forest leading to the discovery of the biggest secret in Smurf history.",null
166426,Pirates of the Caribbean: Dead Men Tell No Tales,Pirates of the Caribbean: Dead Men Tell No Tales,23/05/2017,128,en,Lançado,"Thrust into an all-new adventure, a down-on-his-luck Capt. Jack Sparrow feels the winds of ill-fortune blowing even more strongly when deadly ghost sailors led by his old nemesis, the evil Capt. Salazar, escape from the Devil's Triangle. Jack's only hope of survival lies in seeking out the legendary Trident of Poseidon, but to find it, he must forge an uneasy alliance with a brilliant and beautiful astronomer and a headstrong young man in the British navy.",null
181808,Star Wars: The Last Jedi,Star Wars: The Last Jedi,2017-12-13,152,en,Lançado,"Rey develops her newly discovered abilities with the guidance of Luke Skywalker, who is unsettled by the strength of her powers. Meanwhile, the Resistance prepares to do battle with the First Order.",Darkness rises... and light to meet it
196712,Edge,Edge,03/08/2018,76,en,Lançado,"EDGE. On the surface, it's the story of a madman terrorizing a helpless city, taking lives without a second thought. Blood is this man's water. Enter Jack Rivers, detective for the city's police department and ace headhunter who always gets his man. Stymied by the difficulty of catching this serial murderer, Jack begins to stumble in a blind fury and the line between 'justice' and 'murder' becomes blurred. This is the story of a man whose life is about to change forever. He's about to go over the EDGE.",He's about to go over the EDGE.
245842,The King's Daughter,The King's Daughter,2022-01-21,94,en,Lançado,"King Louis XIV's quest for immortality leads him to capture and steal a mermaid's life force, a move that is further complicated by his illegitimate daughter's discovery of the creature.",What on Earth can hold more power than a king?
248412,Western X,Western X,2016-06-01,90,en,Lançado,A man struggles to find out the truth behind his identity while fighting an evil army.,null


id_filme,release_date
507648,the more Masuda remembers his younger self
881535,northern Sweden. The dream was to get back to Nature
466287,his mother then committed suicide
554860,at least for a brief moment
573364,World's greatest detective. World's smallest package.
767423,"a boy who delivers drugs from being attacked by a group of gangsters in which Song and Nick are the members."""
998516,Manny Pacquiao or fighting the king of the Octagon
452446,in partnership with brave rebel Saudi Arabian undercover reporters


silver.tb_info_filmes | ID único | aprovado=True | chaves duplicadas=0
silver.tb_info_filmes | ID não nulo | aprovado=True | falhas=0/97879
silver.tb_info_filmes | status no domínio | aprovado=True | falhas=0/97879


## 2. Cotação do dólar — `silver.tb_cotacao_dolar`

A tabela Bronze pode não possuir registros para finais de semana e feriados.

Primeiramente é mantida uma única cotação por dia. Em seguida é criado um calendário diário entre a primeira e a última cotação disponível e aplicado **Forward Fill**, fazendo com que dias sem cotação recebam o último valor conhecido.

In [0]:
df_cotacoes = (
    spark.table(f"{catalog}.bronze.tb_cotacao_dolar")
    .select(
        F.to_timestamp("dataHoraCotacao").alias("data_hora_cotacao"),
        F.col("cotacaoCompra").cast("decimal(18,6)").alias("cotacao_compra"),
        F.col("ingestion_datetime").alias("data_ingestao")
    )
    .filter(
        F.col("data_hora_cotacao").isNotNull() &
        F.col("cotacao_compra").isNotNull() &
        (F.col("cotacao_compra") > 0)
    )
    .withColumn("data_cotacao", F.to_date("data_hora_cotacao"))
)
# Caso existam várias cotações para o mesmo dia, mantém a observação
# mais recente considerando horário da cotação e momento da ingestão.
janela_dia = Window.partitionBy("data_cotacao").orderBy(
    F.col("data_hora_cotacao").desc(),
    F.col("data_ingestao").desc()
)

df_cotacao_dia = (
    df_cotacoes
    .withColumn("_ordem", F.row_number().over(janela_dia))
    .filter(F.col("_ordem") == 1)
    .select("data_cotacao", "cotacao_compra")
)

limites = df_cotacao_dia.agg(
    F.min("data_cotacao").alias("data_inicio"),
    F.max("data_cotacao").alias("data_fim")
).first()

if limites["data_inicio"] is None or limites["data_fim"] is None:
    raise ValueError("Não existem cotações válidas na tabela Bronze.")

data_inicio = limites["data_inicio"]
data_fim = limites["data_fim"]

# Série temporal contínua entre a primeira e a última cotação disponível.
# Dias sem cotação (finais de semana/feriados) recebem a última cotação
# conhecida por meio de Forward Fill.

df_calendario = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(data_inicio),
                F.lit(data_fim),
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("data_cotacao")
    )
)

janela_passado = (
    Window.orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_cambio = (
    df_calendario
    .join(df_cotacao_dia, on="data_cotacao", how="left")
    .withColumn(
        "data_cotacao_origem",
        F.last(
            F.when(F.col("cotacao_compra").isNotNull(), F.col("data_cotacao")),
            ignorenulls=True
        ).over(janela_passado)
    )
    .withColumn(
        "cotacao_compra",
        F.last("cotacao_compra", ignorenulls=True).over(janela_passado)
    )
    .select("data_cotacao", "cotacao_compra", "data_cotacao_origem")
)

dq_check_unique(
    "silver.tb_cotacao_dolar",
    "dia único",
    df_cambio,
    ["data_cotacao"]
)

dq_check(
    "silver.tb_cotacao_dolar",
    "cotação positiva",
    df_cambio,
    F.col("cotacao_compra") > 0
)

dq_check(
    "silver.tb_cotacao_dolar",
    "origem da cotação não pode estar no futuro",
    df_cambio,
    F.col("data_cotacao_origem") <= F.col("data_cotacao")
)

(
    df_cambio.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.tb_cotacao_dolar")
)

display(df_cambio.orderBy("data_cotacao"))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


silver.tb_cotacao_dolar | dia único | aprovado=True | chaves duplicadas=0
silver.tb_cotacao_dolar | cotação positiva | aprovado=True | falhas=0/380
silver.tb_cotacao_dolar | origem da cotação não pode estar no futuro | aprovado=True | falhas=0/380


data_cotacao,cotacao_compra,data_cotacao_origem
2023-12-18,4.939300,2023-12-18
2023-12-19,4.865700,2023-12-19
2023-12-20,4.876000,2023-12-20
2023-12-21,4.874900,2023-12-21
2023-12-22,4.861300,2023-12-22
2023-12-23,4.861300,2023-12-22
2023-12-24,4.861300,2023-12-22
2023-12-25,4.861300,2023-12-22
2023-12-26,4.836200,2023-12-26
2023-12-27,4.830000,2023-12-27


## 3. Dados financeiros — `silver.tb_financeiro_filmes`

Os campos de orçamento e receita chegam como texto e podem conter símbolos de moeda, separadores diferentes, abreviações e valores inválidos.

Nesta etapa:

- os campos são higienizados;
- valores inválidos, zerados ou negativos são tratados como ausentes;
- orçamento e receita são convertidos para valores decimais;
- são calculados lucro e margem de lucro;
- os valores em dólares são convertidos para reais utilizando a cotação válida mais recente disponível.

In [0]:
df_fin = spark.table(f"{catalog}.bronze.tb_movies_financials")
# Mantém a versão mais recente de cada filme quando há múltiplas ingestões.
janela_filme = Window.partitionBy("id").orderBy(
    F.col("ingestion_datetime").desc(), F.col("budget"), F.col("revenue")
)
df_fin = df_fin.filter(F.col("id").isNotNull()).withColumn(
    "_ordem", F.row_number().over(janela_filme)
).filter(F.col("_ordem") == 1)

df_fin = df_fin.withColumn(
    # Remove símbolos de moeda e espaços antes da interpretação do valor.
    "budget_limpo",
    F.regexp_replace(F.upper(F.trim("budget")), r"^(USD|US\$|\$)\s*", "")
).withColumn(
    "budget_limpo", F.regexp_replace("budget_limpo", r"\s+", "")
)

df_fin = df_fin.withColumn(
    # Trata tanto formatação no padrão brasileiro (1.000,50)
    # quanto no padrão internacional (1,000.50).
    "budget_limpo",
    F.when(
        F.col("budget_limpo").rlike(r"^-?[0-9]{1,3}(\.[0-9]{3})+(,[0-9]{1,2})?$"),
        F.regexp_replace(F.regexp_replace("budget_limpo", r"\.", ""), ",", ".")
    ).when(
        F.col("budget_limpo").rlike(r"^-?[0-9]{1,3}(,[0-9]{3})+(\.[0-9]{1,2})?$"),
        F.regexp_replace("budget_limpo", ",", "")
    ).otherwise(F.regexp_replace("budget_limpo", ",", "."))
)
df_fin = df_fin.withColumn(
    # Expande abreviações monetárias: K = mil e M = milhão.
    "budget_fator",
    F.when(F.col("budget_limpo").rlike("M$"), 1000000)
     .when(F.col("budget_limpo").rlike("K$"), 1000)
     .otherwise(1)
).withColumn(
    "budget_numero", F.regexp_replace("budget_limpo", "[KM]$", "")
).withColumn(
    "budget_valor",
    F.when(
        F.col("budget_numero").rlike(r"^-?[0-9]{1,12}(\.[0-9]{1,6})?$"),
        F.col("budget_numero").cast("decimal(24,6)") * F.col("budget_fator")
    ).otherwise(None).cast("decimal(28,2)")
).withColumn(
    "orcamento_usd",
    F.when(
        (F.col("budget_valor") > 0) & (F.col("budget_valor") < 10000000000000000),
        F.col("budget_valor").cast("decimal(18,2)")
    ).otherwise(None)
)

df_fin = df_fin.withColumn(
    "revenue_limpo",
    F.regexp_replace(F.upper(F.trim("revenue")), r"^(USD|US\$|\$)\s*", "")
).withColumn(
    "revenue_limpo", F.regexp_replace("revenue_limpo", r"\s+", "")
)

df_fin = df_fin.withColumn(
    "revenue_limpo",
    F.when(
        F.col("revenue_limpo").rlike(r"^-?[0-9]{1,3}(\.[0-9]{3})+(,[0-9]{1,2})?$"),
        F.regexp_replace(F.regexp_replace("revenue_limpo", r"\.", ""), ",", ".")
    ).when(
        F.col("revenue_limpo").rlike(r"^-?[0-9]{1,3}(,[0-9]{3})+(\.[0-9]{1,2})?$"),
        F.regexp_replace("revenue_limpo", ",", "")
    ).otherwise(F.regexp_replace("revenue_limpo", ",", "."))
)
df_fin = df_fin.withColumn(
    "revenue_fator",
    F.when(F.col("revenue_limpo").rlike("M$"), 1000000)
     .when(F.col("revenue_limpo").rlike("K$"), 1000)
     .otherwise(1)
).withColumn(
    "revenue_numero", F.regexp_replace("revenue_limpo", "[KM]$", "")
).withColumn(
    "revenue_valor",
    F.when(
        F.col("revenue_numero").rlike(r"^-?[0-9]{1,12}(\.[0-9]{1,6})?$"),
        F.col("revenue_numero").cast("decimal(24,6)") * F.col("revenue_fator")
    ).otherwise(None).cast("decimal(28,2)")
).withColumn(
    "receita_usd",
    F.when(
        (F.col("revenue_valor") > 0) & (F.col("revenue_valor") < 10000000000000000),
        F.col("revenue_valor").cast("decimal(18,2)")
    ).otherwise(None)
)

df_fin = df_fin.withColumn(
    # O lucro corresponde à diferença entre receita e orçamento.
    # A margem só é calculada quando existe receita positiva, evitando divisão por zero.
    "lucro_usd", (F.col("receita_usd") - F.col("orcamento_usd")).cast("decimal(18,2)")
).withColumn(
    "margem_lucro_percentual",
    F.when(F.col("receita_usd") > 0,
           F.col("lucro_usd") / F.col("receita_usd") * 100)
     .otherwise(None).cast("decimal(18,4)")
)
display(df_fin.select(
    "id", "budget", "revenue", "orcamento_usd", "receita_usd",
    "lucro_usd", "margem_lucro_percentual"
).limit(20))

# Regra de negócio: para converter os valores dos filmes para BRL,
# utiliza-se a cotação de compra mais recente disponível na Silver.

df_taxa_atual = (
    spark.table(f"{catalog}.silver.tb_cotacao_dolar")
    .filter(F.col("cotacao_compra").isNotNull())
    .orderBy(F.col("data_cotacao").desc())
    .select("data_cotacao", "cotacao_compra")
    .limit(1)
)

taxa = df_taxa_atual.first()

if taxa is None:
    raise ValueError("Não há cotação válida disponível para conversão em BRL.")

data_conversao = taxa["data_cotacao"]
cotacao_compra = taxa["cotacao_compra"]

print(f"Cotação usada na conversão: {cotacao_compra} em {data_conversao}")

df_financeiro = (
    df_fin
    .select(
        F.col("id").cast("string").alias("id_filme"),
        "orcamento_usd",
        "receita_usd",
        "lucro_usd",
        "margem_lucro_percentual"
    )
    .withColumn(
        "orcamento_brl",
        (F.col("orcamento_usd") * F.lit(cotacao_compra)).cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        (F.col("receita_usd") * F.lit(cotacao_compra)).cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        (F.col("lucro_usd") * F.lit(cotacao_compra)).cast("decimal(18,2)")
    )
)

dq_check_unique(
    "silver.tb_financeiro_filmes",
    "ID único",
    df_financeiro,
    ["id_filme"]
)

dq_check(
    "silver.tb_financeiro_filmes",
    "orçamento USD positivo ou ausente",
    df_financeiro,
    F.col("orcamento_usd").isNull() | (F.col("orcamento_usd") > 0)
)

dq_check(
    "silver.tb_financeiro_filmes",
    "receita USD positiva ou ausente",
    df_financeiro,
    F.col("receita_usd").isNull() | (F.col("receita_usd") > 0)
)

(
    df_financeiro.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.tb_financeiro_filmes")
)

display(df_financeiro.limit(20))


id,budget,revenue,orcamento_usd,receita_usd,lucro_usd,margem_lucro_percentual
38492,0,0,null,null,null,null
38700,90.0M,426505244,90000000.00,426505244.00,336505244.00,78.8983
42018,0,0,null,null,null,null
42330,0,0,null,null,null,null
45033,337200,0,337200.00,null,null,null
50022,N/A,18850674,null,18850674.00,null,null
66534,0,Não Informado,null,null,null,null
67326,0,0,null,null,null,null
68730,46000000,-23737523,46000000.00,null,null,null
77027,0,0,null,null,null,null


Cotação usada na conversão: 6.191700 em 2024-12-31
silver.tb_financeiro_filmes | ID único | aprovado=True | chaves duplicadas=0
silver.tb_financeiro_filmes | orçamento USD positivo ou ausente | aprovado=True | falhas=0/99006
silver.tb_financeiro_filmes | receita USD positiva ou ausente | aprovado=True | falhas=0/99006


id_filme,orcamento_usd,receita_usd,lucro_usd,margem_lucro_percentual,orcamento_brl,receita_brl,lucro_brl
38492,null,null,null,null,null,null,null
38700,90000000.00,426505244.00,336505244.00,78.8983,557253000.00,2640792519.27,2083539519.27
42018,null,null,null,null,null,null,null
42330,null,null,null,null,null,null,null
45033,337200.00,null,null,null,2087841.24,null,null
50022,null,18850674.00,null,null,null,116717718.21,null
66534,null,null,null,null,null,null,null
67326,null,null,null,null,null,null,null
68730,46000000.00,null,null,null,284818200.00,null,null
77027,null,null,null,null,null,null,null


## 4. Métricas de engajamento — `silver.tb_metricas_engajamento`

As métricas apresentam inconsistências de formatação e registros deslocados para colunas incorretas.

Para evitar falhas ou conversões silenciosas:

- somente valores compatíveis com o formato numérico esperado são convertidos;
- textos e caracteres inválidos tornam-se `NULL`;
- notas TMDB e IMDb são limitadas à escala de 0 a 10;
- popularidade e contagens de votos negativas são invalidadas;
- as contagens são verificadas antes da conversão para `INT`.

In [0]:
df_met = spark.table(f"{catalog}.bronze.tb_movies_metrics")
janela_filme = Window.partitionBy("id").orderBy(
    F.col("ingestion_datetime").desc(), F.col("popularity"), F.col("vote_average"), F.col("vote_count"), F.col("averageRating"), F.col("numVotes")
)
df_met = df_met.filter(F.col("id").isNotNull()).withColumn(
    "_ordem", F.row_number().over(janela_filme)
).filter(F.col("_ordem") == 1)

df_met = df_met.withColumn(
    # Normaliza o separador decimal e converte apenas textos numéricos válidos.
    "popularity_limpo", F.regexp_replace(F.trim("popularity"), ",", ".")
).withColumn(
    "popularidade",
    F.when(F.col("popularity_limpo").rlike(r"^-?[0-9]{1,12}(\.[0-9]{1,8})?$"),
           F.col("popularity_limpo").cast("double")).otherwise(None)
).withColumn(
    "popularidade", F.when(F.col("popularidade") >= 0, F.col("popularidade")).otherwise(None)
)

df_met = df_met.withColumn(
    # Notas fora da escala de negócio de 0 a 10 são consideradas inválidas.
    "vote_average_limpo", F.regexp_replace(F.trim("vote_average"), ",", ".")
).withColumn(
    "nota_media_tmdb",
    F.when(F.col("vote_average_limpo").rlike(r"^-?[0-9]{1,12}(\.[0-9]{1,8})?$"),
           F.col("vote_average_limpo").cast("double")).otherwise(None)
).withColumn(
    "nota_media_tmdb", F.when((F.col("nota_media_tmdb") >= 0) & (F.col("nota_media_tmdb") <= 10), F.col("nota_media_tmdb")).otherwise(None)
)

df_met = df_met.withColumn(

    "averageRating_limpo", F.regexp_replace(F.trim("averageRating"), ",", ".")
).withColumn(
    "nota_media_imdb",
    F.when(F.col("averageRating_limpo").rlike(r"^-?[0-9]{1,12}(\.[0-9]{1,8})?$"),
           F.col("averageRating_limpo").cast("double")).otherwise(None)
).withColumn(
    "nota_media_imdb", F.when((F.col("nota_media_imdb") >= 0) & (F.col("nota_media_imdb") <= 10), F.col("nota_media_imdb")).otherwise(None)
)


df_met = df_met.withColumn(
        # A conversão passa primeiro por BIGINT para evitar overflow durante
        # a leitura; somente valores compatíveis com INT são mantidos.
    "qtd_votos_tmdb_long",
    F.when(F.trim("vote_count").rlike("^[0-9]{1,10}$"),
           F.col("vote_count").cast("bigint")).otherwise(None)
).withColumn(
    "qtd_votos_tmdb",
    F.when(F.col("qtd_votos_tmdb_long") <= 2147483647,
           F.col("qtd_votos_tmdb_long").cast("int")).otherwise(None)
)

df_met = df_met.withColumn(
    "qtd_votos_imdb_long",
    F.when(F.trim("numVotes").rlike("^[0-9]{1,10}$"),
           F.col("numVotes").cast("bigint")).otherwise(None)
).withColumn(
    "qtd_votos_imdb",
    F.when(F.col("qtd_votos_imdb_long") <= 2147483647,
           F.col("qtd_votos_imdb_long").cast("int")).otherwise(None)
)

display(df_met.filter("popularity IS NOT NULL AND popularidade IS NULL")
        .select("id", "popularity").limit(20))
display(df_met.filter("vote_average IS NOT NULL AND nota_media_tmdb IS NULL")
        .select("id", "vote_average").limit(20))

df_metricas = df_met.select(
    F.col("id").cast("string").alias("id_filme"), "popularidade", "nota_media_tmdb",
    "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"
)
(
    df_metricas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.tb_metricas_engajamento")
)

dq_check_unique("silver.tb_metricas_engajamento", "ID único", df_metricas, ["id_filme"])
dq_check("silver.tb_metricas_engajamento", "nota TMDB válida ou ausente", df_metricas,
         F.col("nota_media_tmdb").isNull() |
         ((F.col("nota_media_tmdb") >= 0) & (F.col("nota_media_tmdb") <= 10)))
dq_check("silver.tb_metricas_engajamento", "nota IMDb válida ou ausente", df_metricas,
         F.col("nota_media_imdb").isNull() |
         ((F.col("nota_media_imdb") >= 0) & (F.col("nota_media_imdb") <= 10)))


id,popularity
140300,Po must face two hugely epic
289269,Jane is ready to face off against the deadliest female fighters in the world
321818,but soul crushing job
326549,"""\"""" we see the origins of Torgo"""""""
336435,"""vague and abnormal\"""" and your woman is premenopausal at age 37"""""""
339526,Bron becomes stuck in the Mysterious Beyond. This event urges the young Longneck Littlefoot to take his friends on a quest to the Fire Mountain to save his father. Along the way
355277,a young golden monkey who feels displaced by his baby sister
364257,Mimi's passion for pink
365805,"""\"""" they become the target of a relentless and psychotic trucker who forces them to play a deadly game of cat and mouse"""
367842,"4"""


id,vote_average
66534,52.0
232253,80.0
241259,65.5
283591,73.0
337407,45.309999999999995
339927,65.0
343010,47.76
345909,60.64
349453,88.0
353398,35.0


silver.tb_metricas_engajamento | ID único | aprovado=True | chaves duplicadas=0
silver.tb_metricas_engajamento | nota TMDB válida ou ausente | aprovado=True | falhas=0/99013
silver.tb_metricas_engajamento | nota IMDb válida ou ausente | aprovado=True | falhas=0/99013


## 5. Avaliações dos usuários — `silver.tb_avaliacoes_usuarios`

As avaliações integralmente duplicadas são removidas considerando filme, usuário, nota e comentário.

As notas são convertidas de forma segura e precisam estar entre 0 e 10. Comentários ausentes ou formados somente por espaços são padronizados como `"Sem comentário"`.

In [0]:
# Remove apenas avaliações integralmente duplicadas.
df_avaliacoes = spark.table(f"{catalog}.bronze.tb_movies_reviews")
df_avaliacoes = df_avaliacoes.dropDuplicates(["id", "nome", "nota", "comentario"])
df_avaliacoes = df_avaliacoes.filter("id IS NOT NULL").withColumn(
    # Notas incompatíveis com o formato numérico ou fora de 0–10 tornam-se NULL.
    "nota_limpa", F.regexp_replace(F.trim("nota"), ",", ".")
).withColumn(
    "nota_usuario",
    F.when(F.col("nota_limpa").rlike(r"^-?[0-9]{1,12}(\.[0-9]{1,8})?$"),
           F.col("nota_limpa").cast("double")).otherwise(None)
).withColumn(
    "nota_usuario",
    F.when((F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10),
           F.col("nota_usuario")).otherwise(None)
).withColumn(
    # Ausência de comentário é explicitada para diferenciar um comentário
    # não preenchido de outros valores textuais.
    "comentario_usuario",
    F.when(F.col("comentario").isNull() | (F.trim("comentario") == ""), "Sem comentário")
     .otherwise(F.trim("comentario"))
)
df_avaliacoes = df_avaliacoes.select(
    F.col("id").cast("string").alias("id_filme"),
    F.trim("nome").alias("nome_usuario"),
    "nota_usuario", "comentario_usuario"
)
(
    df_avaliacoes.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.tb_avaliacoes_usuarios")
)
display(df_avaliacoes.limit(10))

dq_check("silver.tb_avaliacoes_usuarios", "comentário preenchido", df_avaliacoes,
         F.col("comentario_usuario").isNotNull() &
         (F.length(F.trim("comentario_usuario")) > 0))
dq_check("silver.tb_avaliacoes_usuarios", "nota válida ou ausente", df_avaliacoes,
         F.col("nota_usuario").isNull() |
         ((F.col("nota_usuario") >= 0) & (F.col("nota_usuario") <= 10)))


id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.4,Sem comentário
1152100,Isabela Pinto 509,7.6,Legal! Uma boa opção para o fim de semana.
846452,Sérgio Freitas,6.1,Assisti até o final mas não me marcou.
410937,Carolina Lopes 276,2.1,Horrível! Perda de tempo.
760653,Maria Barbosa 699,3.5,Sem comentário
1155082,Carolina Almeida 653,3.2,"Fraco, não recomendo."
923930,Carlos Monteiro 465,9.0,Excepcional! História envolvente e atuações impecáveis.
661409,Matheus Gomes 340,9.8,Perfeito! Um dos melhores filmes que já vi.
447163,Natália Freitas 402,9.1,"Obra-prima do cinema, simplesmente espetacular."
464175,Mônica Cavalcanti,7.7,Filme interessante com boas atuações.


silver.tb_avaliacoes_usuarios | comentário preenchido | aprovado=True | falhas=0/32412
silver.tb_avaliacoes_usuarios | nota válida ou ausente | aprovado=True | falhas=0/32412


## 6. Gêneros — `silver.tb_generos`

Como um filme pode possuir vários gêneros armazenados em uma única coluna, os separadores são primeiro padronizados e depois os valores são separados com `split` e `explode`.

Após a explosão, resíduos vazios ou valores fora do domínio conhecido de gêneros são removidos, mantendo uma relação única entre filme e gênero.

In [0]:
df_credits = spark.table(
    f"{bronze_schema}.tb_credits_and_tags"
)

# Domínio utilizado para eliminar resíduos provocados pelo deslocamento
# de colunas existente na base bruta.
generos_validos = [
    "Action", "Adventure", "Animation", "Comedy", "Crime",
    "Documentary", "Drama", "Family", "Fantasy", "History",
    "Horror", "Music", "Mystery", "Romance", "Science Fiction",
    "TV Movie", "Thriller", "War", "Western"
]

# Vírgula e ponto e vírgula são normalizados para um único separador
# antes de transformar cada gênero em uma linha independente.
df_generos = (
    df_credits
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.explode(
            F.split(
                F.regexp_replace(F.col("genres"), ";", ","),
                ","
            )
        ).alias("nome_genero")
    )
    .withColumn("nome_genero", F.trim("nome_genero"))
    .filter(F.col("id_filme").isNotNull())
    .filter(F.col("nome_genero").isNotNull())
    .filter(F.col("nome_genero") != "")
    .filter(F.col("nome_genero").isin(generos_validos))
    .dropDuplicates(["id_filme", "nome_genero"])
)

dq_check_unique(
    "silver.tb_generos",
    "filme e gênero únicos",
    df_generos,
    ["id_filme", "nome_genero"]
)

dq_check(
    "silver.tb_generos",
    "gênero pertence ao domínio",
    df_generos,
    F.col("nome_genero").isin(generos_validos)
)

(
    df_generos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.tb_generos")
)

display(df_generos.limit(20))


silver.tb_generos | filme e gênero únicos | aprovado=True | chaves duplicadas=0
silver.tb_generos | gênero pertence ao domínio | aprovado=True | falhas=0/132775


id_filme,nome_genero
284054,Adventure
315635,Adventure
354912,Family
374720,Action
299537,Science Fiction
324552,Action
398818,Romance
291805,Mystery
508442,Fantasy
497698,Adventure


## 7. Pessoas e empresas — `silver.tb_pessoas_empresas`

A tabela consolida atores, diretores, roteiristas e produtoras em uma única estrutura.

Cada coluna multivalorada é separada em linhas individuais e recebe um `tipo_entidade`. Valores vazios, resíduos numéricos e conteúdos sem caracteres textuais são descartados. Os nomes também são padronizados antes da deduplicação.


In [0]:
def extrair_entidades(df, coluna_origem, tipo_entidade):
    # Função comum para transformar colunas multivaloradas em registros individuais.
    return (
        df
        .select(
            F.col("id").cast("string").alias("id_filme"),
            F.explode(
                F.split(
                    F.regexp_replace(F.col(coluna_origem), ";", ","),
                    ","
                )
            ).alias("nome_entidade")
        )
        .withColumn("nome_entidade", F.trim("nome_entidade"))
        # Remove resíduos numéricos causados pelo Column Shift da origem.
        .filter(F.col("id_filme").isNotNull())
        .filter(F.col("nome_entidade").isNotNull())
        .filter(F.col("nome_entidade") != "")
       
        .filter(~F.col("nome_entidade").rlike(r"^\s*[-+]?\d+([.,]\d+)?\s*$"))
        .filter(F.col("nome_entidade").rlike(r".*[A-Za-zÀ-ÿ].*"))
        .withColumn(
            "nome_entidade",
            # Padroniza capitalização para reduzir duplicidades causadas apenas
            # por diferenças entre maiúsculas e minúsculas.
            F.initcap(F.lower(F.col("nome_entidade")))
        )
        .withColumn("tipo_entidade", F.lit(tipo_entidade))
    )
# O mesmo tratamento é aplicado às quatro categorias,
# diferenciadas pela coluna tipo_entidade.
df_atores = extrair_entidades(df_credits, "cast", "Ator")
df_diretores = extrair_entidades(df_credits, "directors", "Diretor")
df_roteiristas = extrair_entidades(df_credits, "writers", "Roteirista")
df_produtoras = extrair_entidades(df_credits, "production_companies", "Produtora")

df_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
    .dropDuplicates(["id_filme", "nome_entidade", "tipo_entidade"])
)

dq_check_unique(
    "silver.tb_pessoas_empresas",
    "entidade única por filme e tipo",
    df_pessoas_empresas,
    ["id_filme", "nome_entidade", "tipo_entidade"]
)

dq_check(
    "silver.tb_pessoas_empresas",
    "tipo de entidade válido",
    df_pessoas_empresas,
    F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista", "Produtora")
)

dq_check(
    "silver.tb_pessoas_empresas",
    "nome da entidade preenchido",
    df_pessoas_empresas,
    F.col("nome_entidade").isNotNull() &
    (F.length(F.trim("nome_entidade")) > 0)
)

(
    df_pessoas_empresas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.silver.tb_pessoas_empresas")
)

display(df_pessoas_empresas.limit(20))


silver.tb_pessoas_empresas | entidade única por filme e tipo | aprovado=True | chaves duplicadas=0
silver.tb_pessoas_empresas | tipo de entidade válido | aprovado=True | falhas=0/902006
silver.tb_pessoas_empresas | nome da entidade preenchido | aprovado=True | falhas=0/902006


id_filme,nome_entidade,tipo_entidade
283995,Karen Gillan,Ator
209112,Diane Lane,Ator
496243,Park Myung-hoon,Ator
419430,Stephen Root,Ator
424694,Lucy Boynton,Ator
321612,Luke Evans,Ator
429617,Jake Gyllenhaal,Ator
127380,Kaitlin Olson,Ator
343668,Pedro Pascal,Ator
791373,Gal Gadot,Ator


## Validações finais da camada Silver

Após a construção das tabelas são executadas verificações adicionais de qualidade e apresentado um resumo da quantidade de registros gerados em cada tabela.

O resultado das regras permite confirmar se as principais restrições de domínio e unicidade foram respeitadas antes da construção da camada Gold.

In [0]:
dq_check(
    "silver.tb_metricas_engajamento",
    "popularidade não negativa ou ausente",
    df_metricas,
    F.col("popularidade").isNull() | (F.col("popularidade") >= 0)
)

dq_check(
    "silver.tb_metricas_engajamento",
    "votos TMDB não negativos ou ausentes",
    df_metricas,
    F.col("qtd_votos_tmdb").isNull() | (F.col("qtd_votos_tmdb") >= 0)
)

dq_check(
    "silver.tb_metricas_engajamento",
    "votos IMDb não negativos ou ausentes",
    df_metricas,
    F.col("qtd_votos_imdb").isNull() | (F.col("qtd_votos_imdb") >= 0)
)

tabelas_silver = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
    "tb_cotacao_dolar"
]

print("Tabelas Silver geradas:")
for tabela in tabelas_silver:
    df_tabela = spark.table(f"{catalog}.silver.{tabela}")
    print(f"{tabela}: {df_tabela.count()} registros")

if resultados_qualidade:
    df_resultados_qualidade = spark.createDataFrame(resultados_qualidade)
    display(df_resultados_qualidade.orderBy("tabela", "regra"))


silver.tb_metricas_engajamento | popularidade não negativa ou ausente | aprovado=True | falhas=0/99013
silver.tb_metricas_engajamento | votos TMDB não negativos ou ausentes | aprovado=True | falhas=0/99013
silver.tb_metricas_engajamento | votos IMDb não negativos ou ausentes | aprovado=True | falhas=0/99013
Tabelas Silver geradas:
tb_info_filmes: 97879 registros
tb_financeiro_filmes: 99006 registros
tb_metricas_engajamento: 99013 registros
tb_avaliacoes_usuarios: 32412 registros
tb_generos: 132775 registros
tb_pessoas_empresas: 902006 registros
tb_cotacao_dolar: 380 registros


tabela,regra,total_linhas,falhas,unidade_falha,aprovado,verificado_em
silver.tb_avaliacoes_usuarios,comentário preenchido,32412,0,linhas,true,2026-10-07T16:35:54.623Z
silver.tb_avaliacoes_usuarios,nota válida ou ausente,32412,0,linhas,true,2026-10-07T16:35:55.742Z
silver.tb_cotacao_dolar,cotação positiva,380,0,linhas,true,2026-10-07T16:23:16.457Z
silver.tb_cotacao_dolar,dia único,380,0,chaves_duplicadas,true,2026-10-07T16:22:52.149Z
silver.tb_cotacao_dolar,origem da cotação não pode estar no futuro,380,0,linhas,true,2026-10-07T16:23:17.972Z
silver.tb_financeiro_filmes,ID único,99006,0,chaves_duplicadas,true,2026-10-07T16:25:30.148Z
silver.tb_financeiro_filmes,orçamento USD positivo ou ausente,99006,0,linhas,true,2026-10-07T16:25:32.643Z
silver.tb_financeiro_filmes,receita USD positiva ou ausente,99006,0,linhas,true,2026-10-07T16:25:35.085Z
silver.tb_generos,filme e gênero únicos,132775,0,chaves_duplicadas,true,2026-10-07T16:38:27.222Z
silver.tb_generos,gênero pertence ao domínio,132775,0,linhas,true,2026-10-07T16:38:28.280Z
